# 🧪 实验四：模型训练实战

**MobileNetV3 图像分类实战教程**

---

## 实验目标

1. 使用精简配置快速启动训练，观察训练过程
2. 理解各超参数对训练的影响
3. 学会使用预训练 checkpoint 继续训练
4. 分析训练过程中的损失和准确率变化

---

> ⚠️ **注意**：完整训练需要较长时间和 NPU。本实验提供两种模式：
> - **快速模式**（本 Notebook）：使用 5 个 epoch 演示完整训练流程
> - **完整模式**：使用终端命令运行长时间训练
>
> 建议先通过本 Notebook 理解流程，再用完整模式进行实际训练。

In [ ]:
# ====== 1. 导入所需库 ======
# 先设置环境变量抑制警告，子进程（spawn）也会继承
import os
os.environ['PYTHONWARNINGS'] = 'ignore'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from copy import deepcopy
import math
import time
import matplotlib.pyplot as plt

# 导入 torch_npu（Ascend NPU 支持）
import torch_npu

# ====== 自动检测设备：NPU > CPU ======
def get_device():
    if torch.npu.is_available():
        return torch.device('npu:0')
    else:
        return torch.device('cpu')

device = get_device()
print(f"PyTorch 版本: {torch.__version__}")
print(f"使用设备: {device}")

if 'npu' in str(device):
    print(f"   NPU 数量: {torch.npu.device_count()}")
    for i in range(torch.npu.device_count()):
        print(f"     NPU {i}: {torch.npu.get_device_name(i)}")
else:
    print("⚠️  使用 CPU 训练，速度会较慢")

import warnings
warnings.filterwarnings("ignore", message=".*owner does not match.*")
warnings.filterwarnings("ignore", message=".*TASK_QUEUE_ENABLE.*")
warnings.filterwarnings("ignore", message=".*Permission mismatch.*")

In [ ]:
# ====== MobileNetV3 完整模型定义（自包含，不依赖外部文件）======

def get_model_parameters(model):
    total_parameters = 0
    for layer in list(model.parameters()):
        layer_parameter = 1
        for l in list(layer.size()):
            layer_parameter *= l
        total_parameters += layer_parameter
    return total_parameters

def _make_divisible(v, divisor=8, min_value=None):
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v

def _weights_init(m):
    if isinstance(m, nn.Conv2d):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        m.weight.data.fill_(1)
        m.bias.data.zero_()
    elif isinstance(m, nn.Linear):
        n = m.weight.size(1)
        m.weight.data.normal_(0, 0.01)
        m.bias.data.zero_()

class h_sigmoid(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        return F.relu6(x + 3., inplace=self.inplace) / 6.

class h_swish(nn.Module):
    def __init__(self, inplace=True):
        super().__init__()
        self.inplace = inplace
    def forward(self, x):
        out = F.relu6(x + 3., self.inplace) / 6.
        return out * x

class SqueezeBlock(nn.Module):
    def __init__(self, exp_size, divide=4):
        super().__init__()
        self.dense = nn.Sequential(
            nn.Linear(exp_size, exp_size // divide),
            nn.ReLU(inplace=True),
            nn.Linear(exp_size // divide, exp_size),
            h_sigmoid()
        )
    def forward(self, x):
        batch, channels, height, width = x.size()
        out = F.avg_pool2d(x, kernel_size=[height, width]).view(batch, -1)
        out = self.dense(out).view(batch, channels, 1, 1)
        return out * x

class MobileBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernal_size, stride, nonLinear, SE, exp_size):
        super().__init__()
        padding = (kernal_size - 1) // 2
        self.use_connect = stride == 1 and in_channels == out_channels
        self.SE = SE
        activation = nn.ReLU if nonLinear == "RE" else h_swish
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, exp_size, 1, 1, 0, bias=False),
            nn.BatchNorm2d(exp_size), activation(inplace=True))
        self.depth_conv = nn.Sequential(
            nn.Conv2d(exp_size, exp_size, kernal_size, stride, padding, groups=exp_size),
            nn.BatchNorm2d(exp_size))
        if self.SE:
            self.squeeze_block = SqueezeBlock(exp_size)
        self.point_conv = nn.Sequential(
            nn.Conv2d(exp_size, out_channels, 1, 1, 0),
            nn.BatchNorm2d(out_channels), activation(inplace=True))
    def forward(self, x):
        out = self.depth_conv(self.conv(x))
        if self.SE:
            out = self.squeeze_block(out)
        out = self.point_conv(out)
        return x + out if self.use_connect else out

class MobileNetV3(nn.Module):
    def __init__(self, model_mode="LARGE", num_classes=1000, multiplier=1.0, dropout_rate=0.0):
        super().__init__()
        self.num_classes = num_classes
        if model_mode == "LARGE":
            layers = [
                [16, 16, 3, 1, "RE", False, 16], [16, 24, 3, 2, "RE", False, 64],
                [24, 24, 3, 1, "RE", False, 72], [24, 40, 5, 2, "RE", True, 72],
                [40, 40, 5, 1, "RE", True, 120], [40, 40, 5, 1, "RE", True, 120],
                [40, 80, 3, 2, "HS", False, 240], [80, 80, 3, 1, "HS", False, 200],
                [80, 80, 3, 1, "HS", False, 184], [80, 80, 3, 1, "HS", False, 184],
                [80, 112, 3, 1, "HS", True, 480], [112, 112, 3, 1, "HS", True, 672],
                [112, 160, 5, 1, "HS", True, 672], [160, 160, 5, 2, "HS", True, 672],
                [160, 160, 5, 1, "HS", True, 960],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(160 * multiplier), _make_divisible(960 * multiplier), 1, 1),
                nn.BatchNorm2d(_make_divisible(960 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(960 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        elif model_mode == "SMALL":
            layers = [
                [16, 16, 3, 2, "RE", True, 16], [16, 24, 3, 2, "RE", False, 72],
                [24, 24, 3, 1, "RE", False, 88], [24, 40, 5, 2, "RE", True, 96],
                [40, 40, 5, 1, "RE", True, 240], [40, 40, 5, 1, "RE", True, 240],
                [40, 48, 5, 1, "HS", True, 120], [48, 48, 5, 1, "HS", True, 144],
                [48, 96, 5, 2, "HS", True, 288], [96, 96, 5, 1, "HS", True, 576],
                [96, 96, 5, 1, "HS", True, 576],
            ]
            init_conv_out = _make_divisible(16 * multiplier)
            self.init_conv = nn.Sequential(
                nn.Conv2d(3, init_conv_out, 3, 2, 1), nn.BatchNorm2d(init_conv_out), h_swish())
            self.block = nn.Sequential(*[
                MobileBlock(_make_divisible(ic * multiplier), _make_divisible(oc * multiplier),
                           k, s, nl, se, _make_divisible(exp * multiplier))
                for ic, oc, k, s, nl, se, exp in layers])
            self.out_conv1 = nn.Sequential(
                nn.Conv2d(_make_divisible(96 * multiplier), _make_divisible(576 * multiplier), 1, 1),
                SqueezeBlock(_make_divisible(576 * multiplier)),
                nn.BatchNorm2d(_make_divisible(576 * multiplier)), h_swish())
            self.out_conv2 = nn.Sequential(
                nn.Conv2d(_make_divisible(576 * multiplier), _make_divisible(1280 * multiplier), 1, 1),
                h_swish(), nn.Dropout(dropout_rate),
                nn.Conv2d(_make_divisible(1280 * multiplier), num_classes, 1, 1))
        self.apply(_weights_init)
    def forward(self, x):
        out = self.block(self.init_conv(x))
        out = self.out_conv1(out)
        b, c, h, w = out.size()
        return self.out_conv2(F.avg_pool2d(out, [h, w])).view(b, -1)

print("✅ MobileNetV3 模型定义加载完成")

---
## 2. 超参数配置

我们使用简化的超参数进行快速演示：

In [ ]:
# ====== 2. 超参数设置 ======

NUM_EPOCHS = 100           # 训练轮数
BATCH_SIZE = 32           
LEARNING_RATE = 0.001      # 初始学习率
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-5
DROPOUT = 0.2
NUM_CLASSES = 200
MODEL_MODE = "LARGE"
WARMUP_EPOCHS = 5
MIN_LR = 1e-5
DATA_DIR = 'tiny-imagenet-200'
NUM_WORKERS = 1            

print("超参数配置:")
print(f"  模型: MobileNetV3-{MODEL_MODE}")
print(f"  数据集: Tiny ImageNet ({NUM_CLASSES} 类)")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  学习率: {LEARNING_RATE}")
print(f"  Warmup: {WARMUP_EPOCHS} epoch")
print(f"  Dropout: {DROPOUT}")
print(f"  Weight Decay: {WEIGHT_DECAY}")
print(f"  num_workers: {NUM_WORKERS}")

In [ ]:
# ====== 3. 加载数据 ======
print("加载 Tiny ImageNet 数据集...")

# 训练集预处理（RandomResizedCrop 提供尺度多样性，防止过拟合）
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 验证集预处理
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 创建数据集
train_dataset = ImageFolder(
    os.path.join(DATA_DIR, 'train'),
    transform=train_transform
)
val_dataset = ImageFolder(
    os.path.join(DATA_DIR, 'val'),
    transform=val_transform
)

# 多进程数据加载：8 进程 + 512 batch
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=False,
    multiprocessing_context='spawn',
    persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False,
    multiprocessing_context='spawn',
    persistent_workers=True
)

print(f"训练集: {len(train_dataset):,} 张图像 ({len(train_loader)} batches)")
print(f"验证集: {len(val_dataset):,} 张图像 ({len(val_loader)} batches)")
print(f"类别数: {len(train_dataset.classes)}")

In [ ]:
# ====== 4. 创建模型 ======
print("初始化 MobileNetV3 模型...")

model = MobileNetV3(
    model_mode=MODEL_MODE,
    num_classes=NUM_CLASSES,
    multiplier=1.0,
    dropout_rate=DROPOUT
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")


In [ ]:
# ====== 5. 设置优化器、调度器和损失函数 ======

# 选择性权重衰减
def add_weight_decay(model, weight_decay=1e-5):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'bias' in name or 'bn' in name or 'norm' in name:
            no_decay.append(param)
        else:
            decay.append(param)
    return [
        {'params': no_decay, 'weight_decay': 0.},
        {'params': decay, 'weight_decay': weight_decay}
    ]

parameters = add_weight_decay(model, weight_decay=WEIGHT_DECAY)

# RMSprop 优化器（与论文一致）
optimizer = optim.RMSprop(
    parameters,
    lr=LEARNING_RATE,
    alpha=0.9,
    eps=1e-3,
    weight_decay=0,  # 已在 add_weight_decay 中处理
    momentum=MOMENTUM
)

# 余弦退火学习率 + Warmup
class CosineLR:
    def __init__(self, optimizer, total_epochs, warmup_epochs=5, warmup_lr_init=1e-6, min_lr=1e-5):
        self.optimizer = optimizer
        self.total_epochs = total_epochs
        self.warmup_epochs = warmup_epochs
        self.warmup_lr_init = warmup_lr_init
        self.min_lr = min_lr
        self.base_lr = optimizer.param_groups[0]['lr']

    def step(self, epoch):
        if epoch <= self.warmup_epochs:
            lr = self.warmup_lr_init + (self.base_lr - self.warmup_lr_init) * epoch / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        return lr

scheduler = CosineLR(
    optimizer,
    total_epochs=NUM_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS
)

# 标签平滑交叉熵
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, x, target):
        log_probs = torch.nn.functional.log_softmax(x, dim=-1)
        n_classes = x.size(-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))

criterion = LabelSmoothingCrossEntropy(smoothing=0.1)

print("训练组件已就绪:")
print(f"  - 优化器: RMSprop (lr={LEARNING_RATE}, momentum={MOMENTUM})")
print(f"  - 调度器: CosineLR (warmup={WARMUP_EPOCHS}, min_lr={MIN_LR})")
print(f"  - 损失函数: LabelSmoothingCrossEntropy (smoothing=0.1)")

In [ ]:
# ====== 6. 定义训练和验证函数 ======

def accuracy(output, target, topk=(1,)):
    """计算 top-k 准确率"""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, 1, True, True)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            res.append(correct_k.mul_(100.0 / batch_size))
        return res


def train_one_epoch(model, loader, criterion, optimizer, epoch, device):
    """训练一个 epoch"""
    model.train()
    total_loss, total_acc1, total_acc5 = 0, 0, 0
    n_batches = len(loader)
    
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        
        output = model(data)
        loss = criterion(output, target)
        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_acc1 += acc1.item()
        total_acc5 += acc5.item()
        
        if batch_idx % 100 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f"  Train Epoch {epoch} [{batch_idx:>4d}/{n_batches}] "
                  f"Loss: {loss.item():.4f} Acc@1: {acc1.item():.2f}% LR: {lr:.6f}")
    
    return total_loss / n_batches, total_acc1 / n_batches, total_acc5 / n_batches


@torch.no_grad()
def validate(model, loader, criterion, device):
    """验证模型"""
    model.eval()
    total_loss, total_acc1, total_acc5 = 0, 0, 0
    n_batches = len(loader)
    
    for data, target in loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        loss = criterion(output, target)
        acc1, acc5 = accuracy(output, target, topk=(1, 5))
        
        total_loss += loss.item()
        total_acc1 += acc1.item()
        total_acc5 += acc5.item()
    
    return total_loss / n_batches, total_acc1 / n_batches, total_acc5 / n_batches

print("✅ 训练和验证函数已定义")

---
## 7. 开始训练！

> ⏱️ 5 个 epoch 的训练大约需要 5-10 分钟（取决于 NPU）

In [ ]:
# ====== 7. 训练循环 ======
print("=" * 60)
print(f"开始训练 MobileNetV3-{MODEL_MODE} on Tiny ImageNet")
print(f"总共 {NUM_EPOCHS} 个 epoch")
print("=" * 60)

train_losses, val_losses = [], []
train_acc1_list, val_acc1_list = [], []
best_acc1 = 0
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n{'='*40}")
    
    # 更新学习率
    current_lr = scheduler.step(epoch)
    
    # 训练
    train_loss, train_acc1, train_acc5 = train_one_epoch(
        model, train_loader, criterion, optimizer, epoch, device
    )
    
    # 验证
    val_loss, val_acc1, val_acc5 = validate(model, val_loader, criterion, device)
    
    # 记录
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_acc1_list.append(train_acc1)
    val_acc1_list.append(val_acc1)
    
    is_best = val_acc1 > best_acc1
    best_acc1 = max(val_acc1, best_acc1)
    
    print(f"{'='*40}")
    print(f"Epoch {epoch:2d} 总结:")
    print(f"  Train Loss: {train_loss:.4f}  Acc@1: {train_acc1:.2f}%  Acc@5: {train_acc5:.2f}%")
    print(f"  Val   Loss: {val_loss:.4f}  Acc@1: {val_acc1:.2f}%  Acc@5: {val_acc5:.2f}%")
    print(f"  LR: {current_lr:.6f}  Best Acc@1: {best_acc1:.2f}% {'👑' if is_best else ''}")

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"训练完成！总用时: {total_time:.1f}s ({total_time/60:.1f}min)")
print(f"最佳验证 Acc@1: {best_acc1:.2f}%")

In [ ]:
# ====== 8. 可视化训练结果 ======
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, NUM_EPOCHS + 1), train_losses, 'b-o', label='Train Loss', markersize=4)
plt.plot(range(1, NUM_EPOCHS + 1), val_losses, 'r-o', label='Val Loss', markersize=4)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, NUM_EPOCHS + 1), train_acc1_list, 'b-o', label='Train Acc@1', markersize=4)
plt.plot(range(1, NUM_EPOCHS + 1), val_acc1_list, 'r-o', label='Val Acc@1', markersize=4)
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Curve')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📈 观察训练曲线:")
print("  1. 损失是否持续下降？")
print("  2. 验证准确率是否在提升？")
print("  3. 是否存在过拟合（训练损失下降但验证损失上升）？")

---
## 8. 保存模型

训练完成后，保存模型 checkpoint 以便后续评估和推理。

In [ ]:
# ====== 9. 保存模型 Checkpoint ======
import os

checkpoint_dir = os.path.abspath('checkpoint')
os.makedirs(checkpoint_dir, exist_ok=True)

state = {
    'model': model.state_dict(),
    'best_acc1': best_acc1,
    'epoch': NUM_EPOCHS,
}

filename = f'best_model_{MODEL_MODE}_TINY_IMAGENET_demo_ckpt.t7'
save_path = os.path.join(checkpoint_dir, filename)
torch.save(state, save_path)
print(f"模型已保存到: {save_path}")
print(f"最佳验证 Acc@1: {best_acc1:.2f}%")

---
## 9. 完整训练指南

以上是快速演示。要进行完整训练以获得更好的效果，请在终端中运行：

### 单 NPU 完整训练

```bash
# 从项目根目录运行
cd ..
python main.py --dataset-mode TINY_IMAGENET --epochs 200 --batch-size 256 --learning-rate 0.01 --workers 4
```

### 多 NPU 训练（分布式）

```bash
torchrun --nproc_per_node=4 main.py --dataset-mode TINY_IMAGENET --epochs 200 --batch-size 256 --learning-rate 0.01 --workers 4 --distributed
```

### 从 checkpoint 继续训练

```bash
python main.py --dataset-mode TINY_IMAGENET --epochs 300 --batch-size 256 --learning-rate 0.01 --load-pretrained True
```

### 调整训练参数

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">参数</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">建议值</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>--model-mode</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">LARGE / SMALL</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">Large 精度更高，Small 速度更快</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>--epochs</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">200-300</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">更多 epoch 通常更好</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>--batch-size</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">256 (或显存允许的最大值)</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">越大训练越稳定</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>--learning-rate</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">0.01 (Large) / 0.1 (Small)</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">参考论文设置</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>--dropout</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">0.2-0.8</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">小数据集用较大 dropout</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>--multiplier</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">0.5-1.0</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">宽度倍率，越小模型越轻量</td>
    </tr>
  </tbody>
</table>

---
## 📝 实验小结

在本实验中，我们：

1. ✅ 从零开始搭建了完整的训练流程
2. ✅ 配置了 RMSprop 优化器、余弦退火学习率和标签平滑损失
3. ✅ 运行了 5 个 epoch 的快速训练演示
4. ✅ 可视化了训练过程中的损失和准确率变化
5. ✅ 保存了模型 checkpoint
6. ✅ 了解了完整训练的命令和参数

**挑战练习：**
1. 尝试将 `MODEL_MODE` 改为 `SMALL`，对比训练速度和准确率
2. 尝试调整 `BATCH_SIZE`，观察对训练速度的影响
3. 如果时间允许，运行完整 200 epoch 训练，看看最终能达到多少准确率

---


## 课后练习

1. (单选题) 保存 checkpoint 时，保存 state_dict 而不是整个模型的主要原因是？
   - A. 文件更小、结构解耦、便于跨版本加载
   - B. 训练更快
   - C. 精度更高
   - D. 无需保存权重

2. (单选题) 最佳 checkpoint 应依据哪个指标选择？
   - A. 验证集 Top-1
   - B. 最后一个 epoch 的训练 loss
   - C. 训练集准确率
   - D. 随机选择

3. (多选题) 断点续训需要保存？
   - A. model.state_dict()
   - B. optimizer.state_dict()
   - C. scheduler.state_dict()
   - D. epoch/step 与随机种子

4. (多选题) DataLoader num_workers 设置过大可能带来？
   - A. 共享内存/内存压力增大
   - B. worker 启动开销增大
   - C. 数据加载不一定更快
   - D. 训练精度下降

5. (判断题) 每个训练 step 应依次执行 optimizer.zero_grad()、loss.backward()、optimizer.step()。

6. (判断题) 把 model.state_dict() 加载到结构完全不同的模型上一定不会报错。

7. (填空题) 防止上一个 batch 梯度累积到当前 step，应在 backward 前调用 ____。

8. (填空题) 验证阶段使用 ____ 关闭梯度计算，避免保存反向图。

9. (简答题) 为什么验证评估放在每个 epoch 或固定间隔而不是每个 batch？

10. (简答题) 提前停止应同时观察训练 loss 与验证指标，为什么不能只观察训练 loss？

11. (代码设计题) 编写 resume_training 函数：从 checkpoint 恢复 epoch、model、optimizer、scheduler 并继续训练。

12. (单选题) 训练 loss 下降但验证 loss 连续 5 轮上升，首选操作是？
   - A. 回滚到验证最佳 checkpoint 并停止或降低学习率
   - B. 增大学习率
   - C. 删除数据增强
   - D. 增加训练 epoch

13. (多选题) 训练时显存 OOM 的可能原因包括？
   - A. batch_size 过大
   - B. num_workers 过多导致内存峰值高
   - C. 保存了过多中间激活
   - D. 使用 bf16

14. (判断题) torch.no_grad() 下验证会显著降低显存占用，因为不构建反向图。

15. (简答题) 要实现严格可复现训练，需要控制哪些因素？请至少列出 4 项。

> 参考答案见 answer/02.06_training_practice_answer.ipynb。